# 08-3 空間統計分析：登革熱在台灣哪裡「真的」群聚？

前兩個 notebook 教你**畫**空間分布圖（heatmap、spot map、choropleth）。但地圖上出現一片紅，你怎麼知道那是**真的群聚**，還是眼睛自己腦補的？這個 notebook 教你用**空間統計**把「感覺」變成「證據」，回答三個問題：

1. 全臺登革熱到底有沒有空間群聚？ → **全域 Moran's I**
2. 哪些縣市是**熱區核心 / 安全淨土 / 空間離群值**？ → **局部 LISA**
3. 哪裡是統計上顯著的**熱點**？ → **Getis-Ord Gi\***

> 🦟 **換個舞台：為什麼改用登革熱？**
> 退伍軍人病是「一棟樓」的故事，單一建築不適合做縣市級的空間統計。所以這一段換上**登革熱 × 全臺縣市**——這正是台灣空間流病最經典的應用（登革熱熱點年年落在南部）。你學到的方法完全一樣能**縮小尺度**，拿去在一座城市的各棟大樓、各個街廓找 Legionella 熱區。
>
> ⚠️ 本 notebook 的病例數是**合成教學資料**（依台灣南部高、北部低的真實樣態設計），不是實際通報數字。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- 套件與字型設定 ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from libpysal.weights import Queen, KNN          # 空間權重（誰是誰的鄰居）
from esda.moran import Moran, Moran_Local         # 全域 + 局部空間自相關
from esda.getisord import G_Local                 # Getis-Ord Gi* 熱點分析
from esda.smoothing import Empirical_Bayes        # 疾病製圖：EB 平滑

from epi_learning.viz import configure_chinese_font
configure_chinese_font()   # 讓地圖上的中文正常顯示（否則會變成方框 □□□）

## Step 1：載入台灣地圖 + 合成登革熱資料

我們用書中內建的**真實台灣縣市界線** GeoJSON，並依「南部高、北部東部低」的真實樣態，合成一份每個縣市的**登革熱發生率（每十萬人）**。同時記下每個縣市的**人口**和**病例數**——待會兒「小人口不穩定」的陷阱會用到。

In [ ]:
# 讀取真實台灣縣市界線（22 縣市，含離島）
gdf = gpd.read_file("data/geojson/county_smooth_inset.geojson")[
    ["COUNTYNAME", "COUNTYENG", "is_inset", "geometry"]
]

# 各縣市約略人口
pop = {"臺北市":2500000,"新北市":4000000,"桃園市":2270000,"臺中市":2820000,"臺南市":1870000,
       "高雄市":2750000,"基隆市":365000,"新竹市":450000,"嘉義市":265000,"新竹縣":570000,
       "苗栗縣":540000,"彰化縣":1250000,"南投縣":480000,"雲林縣":670000,"嘉義縣":500000,
       "屏東縣":810000,"宜蘭縣":454000,"花蓮縣":320000,"臺東縣":215000,
       "澎湖縣":105000,"金門縣":140000,"連江縣":13000}
# 合成登革熱病例數：南部（臺南/高雄/屏東）爆量，北部東部零星；連江縣人口極少
cases = {"臺南市":2100,"高雄市":2600,"屏東縣":710,"嘉義縣":290,"嘉義市":140,"雲林縣":270,
         "彰化縣":275,"臺中市":505,"南投縣":58,"苗栗縣":54,"桃園市":205,"新竹縣":46,"新竹市":32,
         "臺北市":150,"新北市":240,"基隆市":18,"宜蘭縣":18,"花蓮縣":10,"臺東縣":6,
         "澎湖縣":9,"金門縣":5,"連江縣":7}

gdf["population"] = gdf["COUNTYNAME"].map(pop)
gdf["cases"]      = gdf["COUNTYNAME"].map(cases)
gdf["rate"]       = (gdf["cases"] / gdf["population"] * 100000).round(1)   # 每十萬人發生率

# 先「用眼睛看」：畫原始發生率地圖
fig, ax = plt.subplots(figsize=(6, 7))
gdf.plot(column="rate", cmap="Reds", legend=True, edgecolor="white", linewidth=0.4, ax=ax,
         legend_kwds={"label": "登革熱發生率 (每十萬人)"})
ax.set_title("原始發生率地圖：南部好像一片紅……但那是「真群聚」嗎？")
ax.axis("off")
plt.tight_layout()
plt.show()

## 為什麼不能「只用眼睛」看地圖？

> 🌌 **看星座的陷阱**：人腦是一台「找圖案的機器」——把隨機散布的星星硬連成獵戶座。看著色地圖也一樣：你**一定**會「看到」群聚，但那可能只是隨機的顏色排列。

**空間統計**就是一把尺，量出「這個群聚是真的，還是我腦補的」。它背後有一條地理鐵律——**Tobler 第一定律：「近的東西比較像」**。所以我們要問的不是「有沒有群聚」，而是：

> **「這個相似程度，有沒有超過『隨機本來就會有』的程度？」**

檢定的做法出奇地直白：把各縣市的登革熱數字**剪下來、洗牌、隨機重貼**到地圖上很多次，看真實地圖有沒有比洗出來的更集中。這就是接下來 Moran's I 在做的事。

## Step 2：先定義「鄰居」——空間權重（spatial weights）

要說「跟鄰居很像」之前，得先白紙黑字**定義誰是鄰居**。

> 🎲 **大富翁棋盤比喻**：**Queen 接壤（contiguity）**——只要兩縣市邊界「碰到」（連一個角都算）就是鄰居，像棋盤上相鄰的格子。權重矩陣 $W$ 就是一張「誰是誰的鄰居」的點名表（0/1）。`transform="r"`（row-standardized）= 每個縣市的鄰居權重加起來 = 1，等於「鄰居公平投票，鄰居越多、每票越輕」。

In [ ]:
# Queen 接壤權重（邊界碰到就是鄰居）
w_all = Queen.from_dataframe(gdf, use_index=False)

# ⚠️ 離島的難題：金門、澎湖、連江是海上孤島，用「接壤」定義，一個鄰居都沒有！
islands = [gdf.iloc[i]["COUNTYNAME"] for i in w_all.islands]
print("接壤定義下『沒有鄰居』的縣市：", islands)

看到了嗎？**金門、澎湖、連江**用「接壤」根本找不到鄰居——這叫**孤島（islands）**，空間統計會算不下去。這也順便示範了空間分析的頭號脾氣：**換一種鄰居定義，答案就會變**（稍後「陷阱」會再談）。

兩種常見解法：

- **KNN（k 最近鄰）**：不管接不接壤，直接抓「地理上最近的 k 個」當鄰居——每個縣市都保證有鄰居。
- **聚焦相連的區域**：離島本來就與本島隔海、傳播動態不同，所以下面的**群聚分析我們聚焦在本島 19 個相連的縣市**；離島那幾個「小人口、率不穩」的數字，留到 Step 6 的「平滑」再處理。

In [ ]:
# 聚焦本島 19 個相連縣市（is_inset=False），用 Queen 接壤權重
main = gdf[~gdf["is_inset"]].reset_index(drop=True)
w = Queen.from_dataframe(main, use_index=False)
w.transform = "r"   # row-standardized
print(f"本島縣市數：{len(main)}，平均每個縣市有 {w.mean_neighbors:.1f} 個鄰居，孤島數：{len(w.islands)}")

# （備選：KNN 讓「所有」22 縣市都有鄰居，離島也不例外）
w_knn = KNN.from_dataframe(gdf, k=4)
print(f"若改用 KNN(k=4)：全部 {len(gdf)} 縣市都有鄰居，孤島數：{len(w_knn.islands)}")

## Step 3：全域 Moran's I —— 整張地圖的「物以類聚」指數

**全域 Moran's I** 用**一個數字**總結整座島的空間結構：

- **I ≈ +1**：高的縣市緊挨著高的、低的挨著低的（南部一片紅、北部一片淡）——完美分區。
- **I ≈ 0**：顏色像灑胡椒鹽，高低隨機散布，沒有地域性。
- **I ≈ −1**：像花格子襯衫，高一定挨著低（很罕見）。

光有 I 值還不夠，要追問：**「把數字洗牌 999 次，會不會也洗出這麼高的 I？」** 這個洗牌得到的 **p 值**，才告訴你群聚是不是真的。

In [ ]:
y = main["rate"].values

moran = Moran(y, w, permutations=999)
print(f"全域 Moran's I = {moran.I:.3f}")
print(f"p 值（洗牌 999 次）= {moran.p_sim:.4f}")
print(f"z 分數 = {moran.z_sim:.2f}")

verdict = "有顯著空間群聚 ✅" if moran.p_sim < 0.05 else "看不出空間群聚"
print(f"\n判讀：I = {moran.I:.2f} 且 p < 0.05 → 南部那片紅{verdict}——不是我們眼睛腦補的。")

## Step 4：局部 LISA —— 熱區核心到底在哪？（本段重點）

全域 Moran 給整座島**一個**分數；**LISA（Local Indicators of Spatial Association）**把鏡頭拉近，問**每一個**縣市：「你這一區，跟你的鄰居，是哪一種關係？」它同時看兩件事——**你自己的值** vs **鄰居的平均值**——分成四種鄰里：

![LISA 的四種鄰里](../images/lisa_quadrants.svg)

- **HH 高-高（疫情震央）**：自己高、鄰居也高。你站在火場正中央。→ 核心疫區，全區投入、找共同傳染源。
- **LL 低-低（安全淨土）**：自己低、鄰居也低。→ 低優先，可當對照區。
- **HL 高-低（孤島火苗）**：自己高、鄰居全低（空間離群值）。→ **最該警覺**！可能是新的獨立傳入或資料異常。
- **LH 低-高（颱風眼）**：自己低、鄰居全高（空間離群值）。→ 可能「還沒被傳到」，快防守。

> ⚠️ **esda 的象限編碼**：`.q` 裡 **1=HH、2=LH、3=LL、4=HL**（LH 是 2、不是 3！）貼標籤前務必對照。

In [ ]:
lisa = Moran_Local(y, w, permutations=999, seed=8)

# .q: 1=HH, 2=LH, 3=LL, 4=HL；只保留顯著（p < 0.05）的縣市
labels = {1: "HH 高-高（震央）", 2: "LH 低-高（颱風眼）",
          3: "LL 低-低（淨土）", 4: "HL 高-低（火苗）"}
main["lisa"] = ["不顯著" if p >= 0.05 else labels[q]
                for q, p in zip(lisa.q, lisa.p_sim)]

print("顯著的空間群聚 / 離群值：")
for name, lab, r in zip(main["COUNTYNAME"], main["lisa"], main["rate"]):
    if lab != "不顯著":
        print(f"   {name}：{lab}（發生率 {r}）")

# 畫 LISA 群聚圖（左：原始率；右：LISA 分類）
color = {"HH 高-高（震央）":"#D94452", "LL 低-低（淨土）":"#6A9BCC",
         "HL 高-低（火苗）":"#D97757", "LH 低-高（颱風眼）":"#9FC0E0", "不顯著":"#EDEDED"}
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
main.plot(column="rate", cmap="Reds", legend=True, edgecolor="white", linewidth=0.4, ax=axes[0])
axes[0].set_title("原始發生率"); axes[0].axis("off")
for lab, c in color.items():
    sub = main[main["lisa"] == lab]
    if len(sub):
        sub.plot(ax=axes[1], color=c, edgecolor="white", linewidth=0.4, label=lab)
axes[1].set_title("LISA 群聚圖：紅=震央 藍=淨土 灰=不顯著")
axes[1].axis("off"); axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## Step 5：熱點分析 Getis-Ord Gi\* —— 給長官看的熱區圖

> 🌡️ **紅外線熱像儀比喻**：Gi\* 不管「你跟鄰居像不像」，它只問一件事——把你和你的鄰居**圈成一圈**，這一圈的**總熱度**，有沒有比全國平均**燙得離譜**？輸出是 **z 分數**＝「燙了幾個標準差」。z ≈ +3 → 紅得發燙（99.9% 不是巧合）；z ≈ −3 → 冷到結冰（顯著低）；z ≈ 0 → 體溫正常。

跟 LISA 不同，Gi\* **沒有「唱反調」的離群類別**，它只給你一支溫度計、一條紅到藍的連續光譜——最適合做「哪裡該優先派人」的熱區地圖。

In [ ]:
# Gi* 慣例用二元權重（star=True 代表把自己也算進圈內）
w_b = Queen.from_dataframe(main, use_index=False)
w_b.transform = "B"
gi = G_Local(y, w_b, permutations=999, seed=8, star=True)

# 小樣本用「洗牌 p 值」判斷顯著，再用 z 的正負分冷熱
main["gi_z"] = gi.Zs
hot  = main.loc[(gi.p_sim < 0.05) & (gi.Zs > 0), "COUNTYNAME"].tolist()
cold = main.loc[(gi.p_sim < 0.05) & (gi.Zs < 0), "COUNTYNAME"].tolist()
print("顯著熱點（hot spots）：", hot)
print("顯著冷點（cold spots）：", cold)

fig, ax = plt.subplots(figsize=(6, 7))
main.plot(column="gi_z", cmap="RdBu_r", legend=True, edgecolor="white", linewidth=0.4, ax=ax,
          vmin=-3, vmax=3, legend_kwds={"label": "Gi* z 分數（紅=熱 藍=冷）"})
ax.set_title("Getis-Ord Gi* 熱點圖：南部是統計上顯著的登革熱熱區")
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 6：概念延伸——掃描統計與疾病製圖（平滑）

上面三招（Moran's I / LISA / Gi\*）是縣市層級最常用的。實務上還有兩個更進階的工具，這裡先建立概念（完整實作多半用 R 或專門軟體）：

**① Kulldorff 掃描統計（Scan Statistic）**
> 📡 **雷達畫圈比喻**：在地圖上不斷移動、放大一個圓圈，自動找出「這個圓裡病例特別多、外面正常」的可疑集群。
好處：能抓**不規則、位置未知**的集群，還能做**時間-空間**掃描（何時、何地一起爆）。CDC 監測系統常用它做早期預警。工具：**SaTScan**（免費軟體）、`rsatscan`。

**② 貝氏疾病製圖 / 空間平滑（Disease Mapping / Smoothing）**
> 📷 **修復模糊照片比喻**：小人口的縣市，原始率**忽高忽低**（分母太小）。平滑用「向鄰居借資訊」的方式，估出比較穩定的風險。工具：**R-INLA**、`CARBayes`（BYM 模型）；Python 有 `esda.smoothing`。

**為什麼需要平滑？先看「小人口不穩定」有多可怕：**

In [ ]:
# 連江縣人口只有 1.3 萬——率會因為「多一個案例」就大跳
p_lienchiang = 13000
print("連江縣（人口僅 1.3 萬）原始率有多不穩：")
for c in [6, 7, 8]:
    print(f"   {c} 例 → 發生率 {c / p_lienchiang * 100000:.1f} / 十萬")
print("   → 只差 1 個案例，率就跳動 ~7.7！小人口的原始率極不可靠。\n")

# Empirical Bayes 平滑：向「全體資料估出的先驗」借力，穩定小區域的率
eb = Empirical_Bayes(gdf["cases"].values, gdf["population"].values)
gdf["rate_eb"] = (eb.r * 100000).round(1)
show = ["連江縣", "金門縣", "澎湖縣", "臺南市", "高雄市"]
print("原始率 vs EB 平滑後（人口越小、被修正越多）：")
for n in show:
    r = gdf.loc[gdf["COUNTYNAME"] == n].iloc[0]
    print(f"   {n}：原始 {r['rate']:>6} → EB {r['rate_eb']:>6}（{r['cases']} 例 / {r['population']:,} 人）")

## ⚠️ 空間分析的五個陷阱

1. **MAUP（可變面積單元問題）**：換空間單元（縣市 → 鄉鎮 → 村里），Moran's I 和熱區可能**整個翻掉**。縣市級看到的南部群聚，到村里級可能碎裂或更尖銳。**你的結論，永遠綁在你選的單元上。**
2. **權重定義敏感度**：Queen vs KNN、k=4 vs k=8，結果會移動（我們在 Step 2 已親眼看到離島讓答案改變）。做完務必換一種權重檢查穩不穩。
3. **多重比較**：19 個縣市＝19 次檢定，α=0.05 下光靠運氣就約有 1 個假陽性。看到「只有單一縣市剛好顯著」，要存疑（esda 的 `p_sim` 預設**不做** FDR 校正）。
4. **小人口不穩定**：分母小的地區，原始率極不穩（Step 6 已示範）。小區域一定要考慮**平滑**，別直接畫原始率。
5. **生態謬誤 + 群聚 ≠ 因果**：「南部縣市發生率高」≠「南部每個人風險都高」，更≠「住南部**導致**登革熱」。空間統計只告訴你**去哪裡找**，不告訴你**為什麼**——Moran 顯著不會幫你指出病媒蚊或積水容器。

## 收尾：找到「哪裡」之後呢？

你剛剛把一張「看起來很紅」的地圖，變成了**有統計證據**的結論：

- **全域 Moran's I** 說：全臺登革熱**確實**有空間群聚（不是錯覺）。
- **LISA** 說：**南部（臺南／高雄／嘉義）是熱區核心（HH）**，北部是安全淨土（LL）。
- **Gi\*** 說：南部是統計上顯著的**熱點**，該優先投入孳生源清除與噴藥。

> 🧭 **一樣的方法，縮小尺度就能回到本書的主線**：把「縣市」換成「一座城市裡的各棟大樓 / 各個街廓」，同一套 Moran's I / LISA / Gi\* 就能找出 **Legionella 的建築熱區**。
>
> 但別忘了最後一句：**空間群聚只告訴你「在哪裡」，永遠不告訴你「為什麼」。** 找到熱區之後，真正的答案要靠**現場疫調 + 環境採檢 + 前面幾章的 2×2、迴歸**去追。地圖指路，證據破案。